In [1]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 29.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.6 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 93.5 MB/s eta 0:00:00:00:01


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [3]:
base_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Use your saved checkpoint or final adapter folder
adapter_dir = "/kaggle/input/models/priyansh384/checkpont/pytorch/default/1/llama3-instruction-finetuned/checkpoint-84"
# or:
# adapter_dir = "/kaggle/input/notebook3b7141acda/llama3-instruction-finetuned"

prompt = "त्वचा की मुँहासे की समस्या महिला के लिए पूछा गया, 30 साल पिछले साल मुंबई से पुणे चले गए। तब से, त्वचा का अंधेरा होने और मुँहासे के अंधेरे धब्बे छोड़ने का मुद्दा है।"

In [ ]:
from huggingface_hub import login


login("")

In [5]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

base_model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRM

In [7]:
ft_model = PeftModel.from_pretrained(base_model, adapter_dir)
ft_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [8]:
def generate_answer(model, prompt, tokenizer, max_new_tokens=180):
    messages = [
        {"role": "system", "content": "You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi."},
        {"role": "user", "content": prompt},
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [9]:
print("PROMPT:\n", prompt)
print("\n" + "="*80)

print("\nBASE MODEL OUTPUT:\n")
base_output = generate_answer(base_model, prompt, tokenizer)
print(base_output)

print("\n" + "="*80)

print("\nFINE-TUNED MODEL OUTPUT:\n")
ft_output = generate_answer(ft_model, prompt, tokenizer)
print(ft_output)

PROMPT:
 त्वचा की मुँहासे की समस्या महिला के लिए पूछा गया, 30 साल पिछले साल मुंबई से पुणे चले गए। तब से, त्वचा का अंधेरा होने और मुँहासे के अंधेरे धब्बे छोड़ने का मुद्दा है।


BASE MODEL OUTPUT:



[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

त्वचा की मुँहासे की समस्या महिला के लिए पूछा गया, 30 साल पिछले साल मुंबई से पुणे चले गए। तब से, त्वचा का अंधेरा होने और मुँहासे के अंधेरे धब्बे छोड़ने का मुद्दा है।assistant

नमस्ते, आमतौर पर एक ऐसे डॉक्टर से परामर्श करने की आवश्यकता होती है जो आपको अपने सिर के बारे में मूल्यांकन करेगा और यह निर्धारित करेगा कि क्या आप किस प्रकार के क्षेत्र के लिए इलाज की आवश्यकता हैं , फिर उसी के अनुसार उपचार किया जाएगा।


FINE-TUNED MODEL OUTPUT:

system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

त्वचा की मुँहासे की समस्या महिला के लिए पूछा गया, 30 साल पिछले साल मुंबई से पुणे चले गए। तब से, त्वचा का अंधेरा होने और मुँहासे के अंधेरे धब्बे छोड़ने का मुद्दा है।assistant

आपको एक त्वचा विशेषज्ञ से परामर्श करना चाहिए जिसने आपको कुछ दवाएं देने के बाद आपकी मदद कर सकता है


In [10]:
print("PROMPT:", prompt)
print("\nBASE MODEL:\n", base_output)
print("\nFINE-TUNED MODEL:\n", ft_output)

PROMPT: त्वचा की मुँहासे की समस्या महिला के लिए पूछा गया, 30 साल पिछले साल मुंबई से पुणे चले गए। तब से, त्वचा का अंधेरा होने और मुँहासे के अंधेरे धब्बे छोड़ने का मुद्दा है।

BASE MODEL:
 system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

त्वचा की मुँहासे की समस्या महिला के लिए पूछा गया, 30 साल पिछले साल मुंबई से पुणे चले गए। तब से, त्वचा का अंधेरा होने और मुँहासे के अंधेरे धब्बे छोड़ने का मुद्दा है।assistant

नमस्ते, आमतौर पर एक ऐसे डॉक्टर से परामर्श करने की आवश्यकता होती है जो आपको अपने सिर के बारे में मूल्यांकन करेगा और यह निर्धारित करेगा कि क्या आप किस प्रकार के क्षेत्र के लिए इलाज की आवश्यकता हैं , फिर उसी के अनुसार उपचार किया जाएगा।

FINE-TUNED MODEL:
 system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

त्वचा की मुँहासे की समस्या महिला के लिए पूछा गया, 30 साल पिछले साल मुंबई से पुणे चले गए। तब से, त्वचा का अंधेरा होने और मुँहासे के अंधेरे धब्बे छोड़ने का मुद्दा है।assistant

आपको एक त्वचा 

In [11]:
prompt = "त्वचा और शरीर की देखभाल पुरुष के लिए पूछा गया, 22 साल मैं अपने पूरे शरीर में बुलबुले पा रहा हूं कि गर्मी के कारण कुछ गंभीर है, यह मेरे पूरे शरीर में स्लीप पैदा कर रहा है"

In [12]:
print("PROMPT:\n", prompt)
print("\n" + "="*80)

print("\nBASE MODEL OUTPUT:\n")
base_output = generate_answer(base_model, prompt, tokenizer)
print(base_output)

print("\n" + "="*80)

print("\nFINE-TUNED MODEL OUTPUT:\n")
ft_output = generate_answer(ft_model, prompt, tokenizer)
print(ft_output)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
 त्वचा और शरीर की देखभाल पुरुष के लिए पूछा गया, 22 साल मैं अपने पूरे शरीर में बुलबुले पा रहा हूं कि गर्मी के कारण कुछ गंभीर है, यह मेरे पूरे शरीर में स्लीप पैदा कर रहा है


BASE MODEL OUTPUT:



[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

त्वचा और शरीर की देखभाल पुरुष के लिए पूछा गया, 22 साल मैं अपने पूरे शरीर में बुलबुले पा रहा हूं कि गर्मी के कारण कुछ गंभीर है, यह मेरे पूरे शरीर में स्लीप पैदा कर रहा हैassistant

नमस्कार, गर्मियों के मौसम में, पसीने और बुलबुले एक आम समस्या है। त्वचा विशेषज्ञ द्वारा जांच करें। कृपया उपचार योजना के लिए फोटो भेजें। आशा है कि आपकी मदद करता है।


FINE-TUNED MODEL OUTPUT:

system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

त्वचा और शरीर की देखभाल पुरुष के लिए पूछा गया, 22 साल मैं अपने पूरे शरीर में बुलबुले पा रहा हूं कि गर्मी के कारण कुछ गंभीर है, यह मेरे पूरे शरीर में स्लीप पैदा कर रहा हैassistant

नमस्कार, आपको गर्मी के कारण चेहरे पर बुलबुले होने के कई कारण हैं, जैसे कि त्वचा के अत्यधिक संवेदनशीलता, धूल और वायु के कणों के संपर्क में आना, खराब त्वचा की गुणवत्ता, अत्यधिक सूरज के प्रकाश के संपर्क में आना। चेहरे पर बुलबुले अक्सर एक छोटी सी सूजन या कठ

In [13]:
prompt = "शरीर पर लाल खुजली वाले धब्बे पुरुष के लिए पूछे गए, 31 साल का मैं गर्दन, कंधे, बाहों, हाथों, हाथों के नीचे, छाती और पेट पर कुछ लाल धब्बे रखता हूं। ये धब्बे खुजली वाले हैं और सूरज की रोशनी में बदतर हो जाते हैं! ये पूरे शरीर में फैल रहे हैं, त्वचा के ऊपर और इन धब्बे के पास नाजुक और सूखी है। कृपया सलाह दें कि समस्या क्या हो सकती है।"

In [14]:
print("PROMPT:\n", prompt)
print("\n" + "="*80)

print("\nBASE MODEL OUTPUT:\n")
base_output = generate_answer(base_model, prompt, tokenizer)
print(base_output)

print("\n" + "="*80)

print("\nFINE-TUNED MODEL OUTPUT:\n")
ft_output = generate_answer(ft_model, prompt, tokenizer)
print(ft_output)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
 शरीर पर लाल खुजली वाले धब्बे पुरुष के लिए पूछे गए, 31 साल का मैं गर्दन, कंधे, बाहों, हाथों, हाथों के नीचे, छाती और पेट पर कुछ लाल धब्बे रखता हूं। ये धब्बे खुजली वाले हैं और सूरज की रोशनी में बदतर हो जाते हैं! ये पूरे शरीर में फैल रहे हैं, त्वचा के ऊपर और इन धब्बे के पास नाजुक और सूखी है। कृपया सलाह दें कि समस्या क्या हो सकती है।


BASE MODEL OUTPUT:



[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

शरीर पर लाल खुजली वाले धब्बे पुरुष के लिए पूछे गए, 31 साल का मैं गर्दन, कंधे, बाहों, हाथों, हाथों के नीचे, छाती और पेट पर कुछ लाल धब्बे रखता हूं। ये धब्बे खुजली वाले हैं और सूरज की रोशनी में बदतर हो जाते हैं! ये पूरे शरीर में फैल रहे हैं, त्वचा के ऊपर और इन धब्बे के पास नाजुक और सूखी है। कृपया सलाह दें कि समस्या क्या हो सकती है।assistant

नमस्कार, हाय, आपको एक डॉक्टर से परामर्श करना चाहिए जो आपको एक मुँहासे के प्रकार के बारे में बताएगा, इसलिए आप ठीक से इलाज कर सकते हैं। आप अपने व्यक्तिगत स्थितियों के आधार पर अपने डॉक्टर से किसी भी दवाओं के लिए परामर्श कर सकते हैं। आशा है कि यह मदद करता है। धन्यवाद।


FINE-TUNED MODEL OUTPUT:

system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

शरीर पर लाल खुजली वाले धब्बे पुरुष के लिए पूछे गए, 31 साल का मैं गर्दन, कंधे, बाहों, हाथों, हाथों के नीचे, छाती और पेट पर कुछ लाल धब्बे रखता हूं। ये धब्बे खुजली वाले हैं 

In [15]:
prompt = "नींद की समस्या। पुरुष, 30 साल के लिए पूछा गया पिछले 8 वर्षों में, मेरे पास खर्राट करने की समस्या है। लेकिन पिछले 3 वर्षों में, मेरा खर्राट करने का शोर स्तर बढ़ गया है। मैंने नींद अध्ययन परीक्षण भी किया। इस परिणाम में, नींद की स्कोर 96 है। उस परीक्षण के बाद, मैं वजन घटाने के लिए भोजन को नियंत्रित करता हूं। लेकिन शोर का स्तर नियंत्रित नहीं है। कृपया इस समस्या का समाधान दें।"

In [16]:
print("PROMPT:\n", prompt)
print("\n" + "="*80)

print("\nBASE MODEL OUTPUT:\n")
base_output = generate_answer(base_model, prompt, tokenizer)
print(base_output)

print("\n" + "="*80)

print("\nFINE-TUNED MODEL OUTPUT:\n")
ft_output = generate_answer(ft_model, prompt, tokenizer)
print(ft_output)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
 नींद की समस्या। पुरुष, 30 साल के लिए पूछा गया पिछले 8 वर्षों में, मेरे पास खर्राट करने की समस्या है। लेकिन पिछले 3 वर्षों में, मेरा खर्राट करने का शोर स्तर बढ़ गया है। मैंने नींद अध्ययन परीक्षण भी किया। इस परिणाम में, नींद की स्कोर 96 है। उस परीक्षण के बाद, मैं वजन घटाने के लिए भोजन को नियंत्रित करता हूं। लेकिन शोर का स्तर नियंत्रित नहीं है। कृपया इस समस्या का समाधान दें।


BASE MODEL OUTPUT:



[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

नींद की समस्या। पुरुष, 30 साल के लिए पूछा गया पिछले 8 वर्षों में, मेरे पास खर्राट करने की समस्या है। लेकिन पिछले 3 वर्षों में, मेरा खर्राट करने का शोर स्तर बढ़ गया है। मैंने नींद अध्ययन परीक्षण भी किया। इस परिणाम में, नींद की स्कोर 96 है। उस परीक्षण के बाद, मैं वजन घटाने के लिए भोजन को नियंत्रित करता हूं। लेकिन शोर का स्तर नियंत्रित नहीं है। कृपया इस समस्या का समाधान दें।assistant

हाय, आपको एक यात्रा के लिए जाना चाहिए और वहां आप अपने शरीर के विश्लेषण के लिए एक एयरोस्टेटिस्ट की तलाश कर सकते हैं। यह आपकी मदद करेगा!


FINE-TUNED MODEL OUTPUT:

system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

नींद की समस्या। पुरुष, 30 साल के लिए पूछा गया पिछले 8 वर्षों में, मेरे पास खर्राट करने की समस्या है। लेकिन पिछले 3 वर्षों में, मेरा खर्राट करने का शोर स्तर बढ़ गया है। मैंने नींद अध्ययन परीक्षण भी किया। इस परिणाम में, नींद की स्कोर 96 है। उस परीक्षण के बाद

In [17]:
prompt = "उच्च रक्तचाप को नियंत्रित करने के प्राकृतिक तरीके क्या हैं?"

In [18]:
print("PROMPT:\n", prompt)
print("\n" + "="*80)

print("\nBASE MODEL OUTPUT:\n")
base_output = generate_answer(base_model, prompt, tokenizer)
print(base_output)

print("\n" + "="*80)

print("\nFINE-TUNED MODEL OUTPUT:\n")
ft_output = generate_answer(ft_model, prompt, tokenizer)
print(ft_output)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
 उच्च रक्तचाप को नियंत्रित करने के प्राकृतिक तरीके क्या हैं?


BASE MODEL OUTPUT:



[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

उच्च रक्तचाप को नियंत्रित करने के प्राकृतिक तरीके क्या हैं?assistant

हाय, उच्च रक्तचाप को कम करने के लिए निम्नलिखित चीजें मदद कर सकती हैंः 1. मोटापा से बचें। यदि आप अधिक वजन हैं, तो आप अपने रक्तचाप को बढ़ाते समय मोटापा का अनुभव करते हैं। इसलिए आपको अपना वजन कम करना चाहिए। 2. बहुत पानी पीएं। शराब और कॉफी की तरह कुछ पेय जैसे कि पानी, चाय, आदि का उपयोग करें जो आपके आहार में फाइबर को बढ़ाने में मदद करते हैं। 3. एंटिस्पास्मोडिक दवाएं


FINE-TUNED MODEL OUTPUT:

system

You are a helpful medical assistant. Give safe, clear, and concise answers in Hindi.user

उच्च रक्तचाप को नियंत्रित करने के प्राकृतिक तरीके क्या हैं?assistant

नमस्कार वहाँ, उच्च रक्तचाप की समस्या में कई कारक योगदान देते हैं, इसलिए इसे नियंत्रित करने के लिए एक संयुक्त प्रयास की आवश्यकता होती है। हमारे पास कई घरेलू उपाय हैं जो आपकी मदद कर सकते हैं। निम्नलिखित कुछ सबसे प्रभावी उपाय हैंःः 1. आहारः अपने आहार को बदलें। अधिक फल और सब्